In [1]:
from __future__ import annotations

import json
import re
import wave
from datetime import datetime, timezone
from io import BytesIO
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from google.genai import types
from IPython.display import Audio, display
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "podcast_ai_rag_pipeline"
DOCUMENT_TABLE_ID = "podcast_source_documents"
CHUNK_TABLE_ID = "podcast_source_chunks"
SCRIPT_TABLE_ID = "podcast_script_segments"
ASSET_TABLE_ID = "podcast_assets"

PLANNING_MODEL = "gemini-2.5-pro"
TEXT_MODEL = "gemini-2.5-flash"
TTS_MODEL = "gemini-2.5-flash-tts"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CHUNK_SIZE_CHARS = 1100
CHUNK_OVERLAP_CHARS = 180
MAX_EMBEDDING_TEXT_CHARS = 3000

GCS_AUDIO_PREFIX = "podcast-ai-rag/audio"
GCS_TRANSCRIPT_PREFIX = "podcast-ai-rag/transcripts"
GCS_MANIFEST_PREFIX = "podcast-ai-rag/manifests"
GCS_SUMMARY_PREFIX = "podcast-ai-rag/summaries"

HOST_A_NAME = "Maya"
HOST_B_NAME = "Leon"

HOST_A_VOICE = "Kore"
HOST_B_VOICE = "Puck"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Planning model:", PLANNING_MODEL)
print("Text model:", TEXT_MODEL)
print("TTS model:", TTS_MODEL)
print("Embedding model:", TEXT_EMBEDDING_MODEL)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Planning model: gemini-2.5-pro
Text model: gemini-2.5-flash
TTS model: gemini-2.5-flash-tts
Embedding model: text-embedding-005


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("BigQuery project:", bigquery_client.project)
print("Text embedding model loaded.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
BigQuery project: leafy-guide-497515-m4
Text embedding model loaded.


In [4]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"

try:
    dataset = bigquery_client.create_dataset(dataset_ref)
    print("Created dataset:", dataset.full_dataset_id)
except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)
    print("Dataset already exists:", dataset.full_dataset_id)

document_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{DOCUMENT_TABLE_ID}"
chunk_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
script_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{SCRIPT_TABLE_ID}"
asset_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{ASSET_TABLE_ID}"

print("Document table:", document_table_ref)
print("Chunk table:", chunk_table_ref)
print("Script table:", script_table_ref)
print("Asset table:", asset_table_ref)

Created dataset: leafy-guide-497515-m4:podcast_ai_rag_pipeline
Document table: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_documents
Chunk table: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_chunks
Script table: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_script_segments
Asset table: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_assets


In [5]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)
        print("Table already exists:", table.full_table_id)
        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)
        print("Created table:", created_table.full_table_id)

        return created_table

In [6]:
document_schema = [
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("topic", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

chunk_schema = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("document_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("global_chunk_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("source_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("topic", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("chunk_char_count", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("embedding_model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

script_schema = [
    bigquery.SchemaField("segment_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("episode_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("segment_number", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("speaker", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("voice_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("segment_role", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("script_text", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_chunk_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("audio_gcs_uri", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("mime_type", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("tts_model", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

asset_schema = [
    bigquery.SchemaField("asset_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("episode_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("asset_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("gcs_uri", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("mime_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("metadata_json", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

In [7]:
document_table = ensure_bigquery_table(
    document_table_ref,
    document_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

script_table = ensure_bigquery_table(
    script_table_ref,
    script_schema,
)

asset_table = ensure_bigquery_table(
    asset_table_ref,
    asset_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_documents
Created table: leafy-guide-497515-m4:podcast_ai_rag_pipeline.podcast_source_documents
Table does not exist. Creating: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_chunks
Created table: leafy-guide-497515-m4:podcast_ai_rag_pipeline.podcast_source_chunks
Table does not exist. Creating: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_script_segments
Created table: leafy-guide-497515-m4:podcast_ai_rag_pipeline.podcast_script_segments
Table does not exist. Creating: leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_assets
Created table: leafy-guide-497515-m4:podcast_ai_rag_pipeline.podcast_assets


In [8]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def parse_gcs_uri(gcs_uri: str) -> tuple[str, str]:
    if not gcs_uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got: {gcs_uri}")

    without_scheme = gcs_uri.removeprefix("gs://")
    bucket_name, blob_name = without_scheme.split("/", 1)

    return bucket_name, blob_name


def upload_bytes_to_gcs(
    data: bytes,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def download_gcs_bytes(gcs_uri: str) -> bytes:
    bucket_name, blob_name = parse_gcs_uri(gcs_uri)

    source_bucket = storage_client.bucket(bucket_name)
    blob = source_bucket.blob(blob_name)

    return blob.download_as_bytes()

In [9]:
def display_gcs_audio(gcs_uri: str) -> None:
    audio_bytes = download_gcs_bytes(gcs_uri)

    display(
        Audio(
            data=audio_bytes,
            rate=24000,
        )
    )

In [10]:
def extract_json_object(text: str) -> dict:
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```").strip()

    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in text: {text[:500]}")

    return json.loads(cleaned[start:end + 1])

In [11]:
podcast_brief = """
Create a synthetic internal knowledge base for a technical podcast episode.

The episode should explain how a cloud-native AI system uses:
- Google Cloud Storage for assets
- BigQuery for metadata and vector search
- Vertex AI Gemini for reasoning and script generation
- Vertex AI Text Embeddings for semantic search
- Gemini TTS for cloud-generated audio segments
- chunking for RAG
- structured JSON manifests for downstream automation

The target listener is a Python backend developer learning practical GCP AI engineering.
The podcast should be educational, technical, and portfolio-friendly.
"""

print(podcast_brief)


Create a synthetic internal knowledge base for a technical podcast episode.

The episode should explain how a cloud-native AI system uses:
- Google Cloud Storage for assets
- BigQuery for metadata and vector search
- Vertex AI Gemini for reasoning and script generation
- Vertex AI Text Embeddings for semantic search
- Gemini TTS for cloud-generated audio segments
- chunking for RAG
- structured JSON manifests for downstream automation

The target listener is a Python backend developer learning practical GCP AI engineering.
The podcast should be educational, technical, and portfolio-friendly.



In [12]:
source_document_specs = [
    {
        "document_number": 1,
        "source_type": "architecture_note",
        "topic": "cloud_architecture",
        "title": "Architecture Note: Cloud-Native Podcast AI Pipeline",
        "focus": "GCS asset storage, BigQuery metadata, Vertex AI orchestration, no local media files",
    },
    {
        "document_number": 2,
        "source_type": "engineering_design",
        "topic": "rag_and_chunking",
        "title": "Engineering Design: Chunking and Semantic Search for Podcast Context",
        "focus": "text normalization, chunk size, overlap, embeddings, BigQuery VECTOR_SEARCH",
    },
    {
        "document_number": 3,
        "source_type": "data_contract",
        "topic": "structured_outputs",
        "title": "Data Contract: Podcast Script Segments and Manifest JSON",
        "focus": "episode_id, segment_id, source chunks, speaker roles, GCS URIs, BigQuery rows",
    },
    {
        "document_number": 4,
        "source_type": "operations_guide",
        "topic": "cloud_execution",
        "title": "Operations Guide: Running the Podcast Pipeline Without Local Files",
        "focus": "in-memory audio bytes, direct upload to GCS, avoiding local writes, BigQuery lineage",
    },
    {
        "document_number": 5,
        "source_type": "lesson_notes",
        "topic": "developer_education",
        "title": "Lesson Notes: Teaching RAG Through an AI Podcast Use Case",
        "focus": "how to explain chunking, retrieval, grounding, prompt engineering, and cloud-native asset handling",
    },
]

pd.DataFrame(source_document_specs)

,document_number,source_type,topic,title,focus
0,1,architecture_note,cloud_architecture,Architecture Note: Cloud-Native Podcast AI Pip...,"GCS asset storage, BigQuery metadata, Vertex A..."
1,2,engineering_design,rag_and_chunking,Engineering Design: Chunking and Semantic Sear...,"text normalization, chunk size, overlap, embed..."
2,3,data_contract,structured_outputs,Data Contract: Podcast Script Segments and Man...,"episode_id, segment_id, source chunks, speaker..."
3,4,operations_guide,cloud_execution,Operations Guide: Running the Podcast Pipeline...,"in-memory audio bytes, direct upload to GCS, a..."
4,5,lesson_notes,developer_education,Lesson Notes: Teaching RAG Through an AI Podca...,"how to explain chunking, retrieval, grounding,..."


In [13]:
def generate_source_document(spec: dict) -> str:
    prompt = f"""
You are a senior cloud AI engineer writing internal technical documentation.

Podcast brief:
{podcast_brief}

Document metadata:
{json.dumps(spec, indent=2, ensure_ascii=False)}

Write a realistic internal document in English with these sections:
1. Context
2. Architecture
3. Data flow
4. Engineering details
5. Failure modes
6. Best practices
7. Notes for Python implementation

Requirements:
- Make it synthetic but realistic.
- Do not mention that it is AI-generated.
- Include enough content to split into semantic chunks.
- Do not use markdown tables.
- Make the writing concrete and useful for a technical podcast script.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    return response.text.strip()

In [14]:
source_documents = []

for spec in source_document_specs:
    print("=" * 100)
    print("Generating source document:", spec["title"])

    content = generate_source_document(spec)

    document = {
        "document_id": str(uuid4()),
        "document_number": spec["document_number"],
        "source_type": spec["source_type"],
        "topic": spec["topic"],
        "title": spec["title"],
        "content": content,
        "summary": content[:700],
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    source_documents.append(document)

    print("Characters:", len(content))
    print(content[:600])

Generating source document: Architecture Note: Cloud-Native Podcast AI Pipeline
Characters: 10771
{
  "document_number": 1,
  "source_type": "architecture_note",
  "topic": "cloud_architecture",
  "title": "Architecture Note: Cloud-Native Podcast AI Pipeline",
  "focus": "GCS asset storage, BigQuery metadata, Vertex AI orchestration, no local media files"
}

### 1. Context

This document outlines the architecture for "Project Vox," an internal initiative to create a fully cloud-native pipeline for generating synthetic technical podcast episodes. The primary goal is to leverage Google Cloud services to automate the content creation process from raw source material to a final, ready-to-assem
Generating source document: Engineering Design: Chunking and Semantic Search for Podcast Context
Characters: 12844
```json
{
  "document_number": 2,
  "source_type": "engineering_design",
  "topic": "rag_and_chunking",
  "title": "Engineering Design: Chunking and Semantic Search for Podcast Context",

In [15]:
documents_df = pd.DataFrame(
    [
        {
            "document_number": document["document_number"],
            "source_type": document["source_type"],
            "topic": document["topic"],
            "title": document["title"],
            "content_length": len(document["content"]),
        }
        for document in source_documents
    ]
)

documents_df

,document_number,source_type,topic,title,content_length
0,1,architecture_note,cloud_architecture,Architecture Note: Cloud-Native Podcast AI Pip...,10771
1,2,engineering_design,rag_and_chunking,Engineering Design: Chunking and Semantic Sear...,12844
2,3,data_contract,structured_outputs,Data Contract: Podcast Script Segments and Man...,10711
3,4,operations_guide,cloud_execution,Operations Guide: Running the Podcast Pipeline...,10829
4,5,lesson_notes,developer_education,Lesson Notes: Teaching RAG Through an AI Podca...,12784


In [16]:
def normalize_text(text: str) -> str:
    return " ".join(text.split())


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []
    start = 0

    while start < len(clean_text):
        end = min(start + chunk_size_chars, len(clean_text))

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(". ", start, end)

            if sentence_boundary > start + int(chunk_size_chars * 0.6):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(0, end - chunk_overlap_chars)

    return chunks

In [17]:
chunk_rows_without_embeddings = []
global_chunk_number = 1

for document in source_documents:
    chunks = split_text_into_chunks(document["content"])

    print("=" * 100)
    print("Document:", document["title"])
    print("Chunks:", len(chunks))

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "document_id": document["document_id"],
                "document_number": document["document_number"],
                "chunk_number": chunk_index,
                "global_chunk_number": global_chunk_number,
                "source_type": document["source_type"],
                "topic": document["topic"],
                "title": document["title"],
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": TEXT_EMBEDDING_MODEL,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        global_chunk_number += 1

print("Total chunks:", len(chunk_rows_without_embeddings))

chunks_preview_df = pd.DataFrame(
    [
        {
            "global_chunk_number": row["global_chunk_number"],
            "topic": row["topic"],
            "source_type": row["source_type"],
            "chunk_char_count": row["chunk_char_count"],
            "preview": row["chunk_text"][:180],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(10)

Document: Architecture Note: Cloud-Native Podcast AI Pipeline
Chunks: 13
Document: Engineering Design: Chunking and Semantic Search for Podcast Context
Chunks: 15
Document: Data Contract: Podcast Script Segments and Manifest JSON
Chunks: 12
Document: Operations Guide: Running the Podcast Pipeline Without Local Files
Chunks: 12
Document: Lesson Notes: Teaching RAG Through an AI Podcast Use Case
Chunks: 14
Total chunks: 66


,global_chunk_number,topic,source_type,chunk_char_count,preview
0,1,cloud_architecture,architecture_note,1005,"{ ""document_number"": 1, ""source_type"": ""archit..."
1,2,cloud_architecture,architecture_note,982,"assets and a manifest file, enabling downstrea..."
2,3,cloud_architecture,architecture_note,994,ding. * **BigQuery:** Serves a dual purpose. 1...
3,4,cloud_architecture,architecture_note,1077,less compute and orchestration layer. These se...
4,5,cloud_architecture,architecture_note,982,ng. 5. **Persistence:** The function performs ...
5,6,cloud_architecture,architecture_note,1040,N most semantically similar text chunks from t...
6,7,cloud_architecture,architecture_note,667,"nique, predictable name (e.g., `episode_123_se..."
7,8,cloud_architecture,architecture_note,1088,"p. A target chunk size of ~500 tokens is used,..."
8,9,cloud_architecture,architecture_note,1012,"Trade-offs"", ""segments"": [ { ""segment_index"": ..."
9,10,cloud_architecture,architecture_note,994,** The model may generate content not supporte...


In [18]:
def shorten_text_for_embedding(
    text: str,
    *,
    max_chars: int = MAX_EMBEDDING_TEXT_CHARS,
) -> str:
    clean_text = normalize_text(text)

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_text_embedding(text: str) -> list[float]:
    safe_text = shorten_text_for_embedding(text)

    embeddings = text_embedding_model.get_embeddings(
        [safe_text]
    )

    return list(embeddings[0].values)

In [19]:
chunk_rows = []

for row in chunk_rows_without_embeddings:
    print(
        "Embedding chunk:",
        row["global_chunk_number"],
        "|",
        row["topic"],
        "|",
        row["source_type"],
    )

    embedding = get_text_embedding(row["chunk_text"])

    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

    print("Embedding dim:", len(embedding))
    print("Chunk chars:", row["chunk_char_count"])
    print("-" * 100)

print("Chunk rows with embeddings:", len(chunk_rows))

Embedding chunk: 1 | cloud_architecture | architecture_note
Embedding dim: 768
Chunk chars: 1005
----------------------------------------------------------------------------------------------------
Embedding chunk: 2 | cloud_architecture | architecture_note
Embedding dim: 768
Chunk chars: 982
----------------------------------------------------------------------------------------------------
Embedding chunk: 3 | cloud_architecture | architecture_note
Embedding dim: 768
Chunk chars: 994
----------------------------------------------------------------------------------------------------
Embedding chunk: 4 | cloud_architecture | architecture_note
Embedding dim: 768
Chunk chars: 1077
----------------------------------------------------------------------------------------------------
Embedding chunk: 5 | cloud_architecture | architecture_note
Embedding dim: 768
Chunk chars: 982
----------------------------------------------------------------------------------------------------
Embedding chu

In [20]:
document_rows = []

for document in source_documents:
    document_rows.append(
        {
            "document_id": document["document_id"],
            "document_number": document["document_number"],
            "source_type": document["source_type"],
            "topic": document["topic"],
            "title": document["title"],
            "content": document["content"],
            "summary": document["summary"],
            "created_at": document["created_at"],
        }
    )

bigquery_client.query(
    f"TRUNCATE TABLE `{document_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{chunk_table_ref}`",
    location=dataset.location,
).result()

document_errors = bigquery_client.insert_rows_json(
    document_table_ref,
    document_rows,
)

chunk_errors = bigquery_client.insert_rows_json(
    chunk_table_ref,
    chunk_rows,
)

if document_errors:
    print("Document insert errors:")
    print(document_errors)
else:
    print("Inserted document rows:", len(document_rows))

if chunk_errors:
    print("Chunk insert errors:")
    print(chunk_errors)
else:
    print("Inserted chunk rows:", len(chunk_rows))

Inserted document rows: 5
Inserted chunk rows: 66


In [21]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Document count:", result.document_count)
print("Chunk count:", result.chunk_count)
print("Embedding length:", result.embedding_length)

Document count: 5
Chunk count: 66
Embedding length: 768


In [22]:
def search_podcast_chunks(
    query: str,
    *,
    top_k: int = 8,
    topic: str | None = None,
    source_type: str | None = None,
) -> list[dict]:
    query_embedding = get_text_embedding(query)

    filters = []
    query_parameters = [
        bigquery.ArrayQueryParameter(
            "query_embedding",
            "FLOAT64",
            query_embedding,
        ),
        bigquery.ScalarQueryParameter(
            "top_k",
            "INT64",
            top_k,
        ),
    ]

    if topic is not None:
        filters.append("topic = @topic")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "topic",
                "STRING",
                topic,
            )
        )

    if source_type is not None:
        filters.append("source_type = @source_type")
        query_parameters.append(
            bigquery.ScalarQueryParameter(
                "source_type",
                "STRING",
                source_type,
            )
        )

    where_sql = ""

    if filters:
        where_sql = "WHERE " + " AND ".join(filters)

    sql = f"""
    SELECT
      base.chunk_id,
      base.document_id,
      base.document_number,
      base.chunk_number,
      base.global_chunk_number,
      base.source_type,
      base.topic,
      base.title,
      base.chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{chunk_table_ref}`
        {where_sql}
      ),
      'embedding',
      (SELECT @query_embedding AS embedding),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=query_parameters,
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "topic": row.topic,
            "title": row.title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [23]:
query = """
Explain how this podcast pipeline avoids saving local audio files,
uses GCS for assets, BigQuery for metadata, and semantic chunks for RAG.
"""

search_results = search_podcast_chunks(
    query,
    top_k=8,
)

print("Query:")
print(query)

print("\nSearch results:", len(search_results))

for result in search_results:
    print("=" * 100)
    print("Distance:", round(result["distance"], 4))
    print("Topic:", result["topic"])
    print("Source:", result["source_type"])
    print("Document:", result["title"])
    print("Global chunk:", result["global_chunk_number"])
    print(result["chunk_text"][:900])

Query:

Explain how this podcast pipeline avoids saving local audio files,
uses GCS for assets, BigQuery for metadata, and semantic chunks for RAG.


Search results: 8
Distance: 0.1764
Topic: structured_outputs
Source: data_contract
Document: Data Contract: Podcast Script Segments and Manifest JSON
Global chunk: 34
e_id>/segments/<segment_id>.mp3`). 6. **Manifest Creation:** Once all audio segments are generated and their GCS URIs are confirmed, the final `episode_manifest.json` is assembled. This file combines the script text from Gemini with the GCS URIs of the corresponding audio files and the `source_chunk_ids` used to generate the text. This manifest is the final deliverable of the pipeline and is saved to the episode's GCS directory. ### 4. Engineering Details **BigQuery Chunks Table Schema:** The `podcast_knowledge.chunks` table is central to our RAG implementation. - `chunk_id`: STRING (UUID, primary key) - `episode_id`: STRING (Foreign key to an episodes table) - `source_docum

In [24]:
def build_podcast_context(results: list[dict]) -> str:
    parts = []

    for index, result in enumerate(results, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Chunk ID: {result['chunk_id']}",
                    f"Document title: {result['title']}",
                    f"Source type: {result['source_type']}",
                    f"Topic: {result['topic']}",
                    f"Global chunk number: {result['global_chunk_number']}",
                    f"Distance: {result['distance']:.4f}",
                    f"Chunk text: {result['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [25]:
def generate_podcast_script(
    episode_topic: str,
    *,
    top_k: int = 10,
) -> dict:
    results = search_podcast_chunks(
        episode_topic,
        top_k=top_k,
    )

    context = build_podcast_context(results)

    prompt = f"""
You are a technical podcast producer.

Create a short two-host podcast episode script based only on the retrieved source chunks.

Episode topic:
{episode_topic}

Hosts:
- {HOST_A_NAME}: senior cloud AI engineer, precise and practical
- {HOST_B_NAME}: Python backend developer, curious and asks implementation questions

Retrieved source chunks:
{context}

Return only valid JSON:

{{
  "episode_title": "short title",
  "episode_summary": "short summary",
  "segments": [
    {{
      "segment_number": 1,
      "speaker": "{HOST_A_NAME}",
      "segment_role": "intro | explanation | question | example | warning | recap | outro",
      "script_text": "spoken podcast line, max 650 characters",
      "source_numbers": [1, 2]
    }}
  ]
}}

Rules:
- Create exactly 10 segments.
- Alternate speakers naturally.
- Every segment must be short enough for TTS.
- Keep it technical but conversational.
- Explain chunking, embeddings, semantic search, GCS, BigQuery, and TTS.
- Reference source_numbers from the retrieved chunks.
- Do not use markdown.
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
    )

    script_data = extract_json_object(response.text)

    return {
        "script_data": script_data,
        "search_results": results,
        "rag_context": context,
    }

In [26]:
episode_topic = """
Build a cloud-native AI podcast pipeline with document chunking,
semantic search, BigQuery VECTOR_SEARCH, Gemini script generation,
Gemini TTS, and audio stored directly in Google Cloud Storage.
"""

podcast_script_result = generate_podcast_script(
    episode_topic,
    top_k=10,
)

print(json.dumps(podcast_script_result["script_data"], indent=2, ensure_ascii=False))

{
  "episode_title": "Cloud-Native AI Podcast Pipeline",
  "episode_summary": "An overview of an automated, serverless pipeline that uses Gemini, BigQuery Vector Search, and GCS to generate podcast episodes from technical documents.",
  "segments": [
    {
      "segment_number": 1,
      "speaker": "Maya",
      "segment_role": "intro",
      "script_text": "Welcome, everyone. Today we're breaking down our internal 'Project Vox' initiative. The goal was to build a fully cloud-native pipeline to generate synthetic technical podcasts. Crucially, the entire system avoids any local media storage or manual processing, creating a repeatable, serverless, and scalable workflow.",
      "source_numbers": [
        8
      ]
    },
    {
      "segment_number": 2,
      "speaker": "Leon",
      "segment_role": "question",
      "script_text": "That sounds powerful. So, where does it start? If I'm a developer, how do I get from my design documents to a finished audio package? Is the whole thing 

In [27]:
def source_numbers_to_chunk_ids(
    source_numbers: list[int],
    search_results: list[dict],
) -> list[str]:
    chunk_ids = []

    for source_number in source_numbers:
        index = int(source_number) - 1

        if 0 <= index < len(search_results):
            chunk_ids.append(search_results[index]["chunk_id"])

    return chunk_ids

In [28]:
episode_id = str(uuid4())

script_segments_without_audio = []

for segment in podcast_script_result["script_data"]["segments"]:
    speaker = segment["speaker"]

    if speaker == HOST_A_NAME:
        voice_name = HOST_A_VOICE
    elif speaker == HOST_B_NAME:
        voice_name = HOST_B_VOICE
    else:
        voice_name = HOST_A_VOICE

    source_chunk_ids = source_numbers_to_chunk_ids(
        segment.get("source_numbers", []),
        podcast_script_result["search_results"],
    )

    script_segments_without_audio.append(
        {
            "segment_id": str(uuid4()),
            "episode_id": episode_id,
            "segment_number": int(segment["segment_number"]),
            "speaker": speaker,
            "voice_name": voice_name,
            "segment_role": segment.get("segment_role"),
            "script_text": segment["script_text"],
            "source_chunk_ids": source_chunk_ids,
            "audio_gcs_uri": None,
            "mime_type": None,
            "tts_model": None,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )

pd.DataFrame(script_segments_without_audio)

,segment_id,episode_id,segment_number,speaker,voice_name,segment_role,script_text,source_chunk_ids,audio_gcs_uri,mime_type,tts_model,created_at
0,176e1ad0-f73f-4ed3-99db-7056f9841758,b533e3cd-4846-40c9-bceb-a401653df411,1,Maya,Kore,intro,"Welcome, everyone. Today we're breaking down o...",[c0354566-4595-4877-8799-88b67713bd6c],None,None,None,2026-07-01T15:30:05.535013+00:00
1,89045a83-6084-4efa-b3f2-4365cbd084e4,b533e3cd-4846-40c9-bceb-a401653df411,2,Leon,Puck,question,"That sounds powerful. So, where does it start?...","[215e4a84-544f-4720-b968-02c8a189734f, c035456...",None,None,None,2026-07-01T15:30:05.535060+00:00
2,ea0842f6-e4a9-4c72-9442-ac7e7098ac1c,b533e3cd-4846-40c9-bceb-a401653df411,3,Maya,Kore,explanation,It's event-driven. The process kicks off when ...,[7df166c4-c151-4e35-93b5-8c10e691ca49],None,None,None,2026-07-01T15:30:05.535082+00:00
3,e49b1180-9029-49be-808a-26220460bcb6,b533e3cd-4846-40c9-bceb-a401653df411,4,Leon,Puck,question,"Okay, so you have small chunks of text. How do...",[d3b47681-d1b8-4a1e-a18e-5ba41332b28d],None,None,None,2026-07-01T15:30:05.535100+00:00
4,fd7343b4-c645-4455-8353-f7f6a43dfb83,b533e3cd-4846-40c9-bceb-a401653df411,5,Maya,Kore,explanation,That's where embeddings come in. We use a mode...,"[5784b0f1-dacd-48c0-8292-6973fdc3683c, f1d4abe...",None,None,None,2026-07-01T15:30:05.535118+00:00
5,a82c39ea-88f6-4854-a6fe-63d0402879c5,b533e3cd-4846-40c9-bceb-a401653df411,6,Leon,Puck,question,I see. So all the context is sitting in BigQue...,"[4da4bdbd-cdda-40ec-a5a3-df6b01b00534, d3b4768...",None,None,None,2026-07-01T15:30:05.535135+00:00
6,aba92224-07b6-4a90-9f45-6ac98c294973,b533e3cd-4846-40c9-bceb-a401653df411,7,Maya,Kore,explanation,The system generates an embedding for your top...,"[4da4bdbd-cdda-40ec-a5a3-df6b01b00534, f1d4abe...",None,None,None,2026-07-01T15:30:05.535152+00:00
7,73c8f26a-4acf-49b8-ad5a-169e467b1e44,b533e3cd-4846-40c9-bceb-a401653df411,8,Leon,Puck,question,"Alright, so that's the retrieval part of RAG. ...",[bed505ad-3143-4a8e-aadb-6278fd950f02],None,None,None,2026-07-01T15:30:05.535169+00:00
8,c46e56a9-50b8-4186-8022-915644df9f00,b533e3cd-4846-40c9-bceb-a401653df411,9,Maya,Kore,explanation,"We construct a prompt for Gemini Pro, combinin...","[215e4a84-544f-4720-b968-02c8a189734f, bed505a...",None,None,None,2026-07-01T15:30:05.535202+00:00
9,70277b48-e20e-4af2-a93b-58c3e3088031,b533e3cd-4846-40c9-bceb-a401653df411,10,Leon,Puck,recap,"So the final output isn't one big audio file, ...","[215e4a84-544f-4720-b968-02c8a189734f, f360d59...",None,None,None,2026-07-01T15:30:05.535221+00:00


In [29]:
def pcm_to_wav_bytes(
    pcm_data: bytes,
    *,
    channels: int = 1,
    rate: int = 24000,
    sample_width: int = 2,
) -> bytes:
    buffer = BytesIO()

    with wave.open(buffer, "wb") as wav_file:
        wav_file.setnchannels(channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(rate)
        wav_file.writeframes(pcm_data)

    buffer.seek(0)

    return buffer.getvalue()


def extract_audio_bytes_from_response(response) -> bytes | None:
    try:
        parts = response.candidates[0].content.parts
    except Exception:
        return None

    for part in parts:
        inline_data = getattr(part, "inline_data", None)

        if inline_data is None:
            continue

        data = getattr(inline_data, "data", None)

        if data:
            return data

    return None


def sanitize_slug(text: str) -> str:
    lowered = text.lower()
    cleaned = re.sub(r"[^a-z0-9]+", "-", lowered)
    cleaned = cleaned.strip("-")

    return cleaned[:80] or "podcast-segment"

In [30]:
def generate_tts_audio_segment(
    *,
    text: str,
    voice_name: str,
    blob_name: str,
) -> str:
    response = google_vertex_client.models.generate_content(
        model=TTS_MODEL,
        contents=f"Say in a natural technical podcast voice: {text}",
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=voice_name,
                    )
                )
            ),
        ),
    )

    pcm_audio = extract_audio_bytes_from_response(response)

    if not pcm_audio:
        raise RuntimeError("No audio bytes returned from TTS model.")

    wav_audio = pcm_to_wav_bytes(pcm_audio)

    gcs_uri = upload_bytes_to_gcs(
        wav_audio,
        blob_name=blob_name,
        content_type="audio/wav",
    )

    return gcs_uri

In [31]:
script_segments = []

episode_title_slug = sanitize_slug(
    podcast_script_result["script_data"]["episode_title"]
)

for segment in script_segments_without_audio:
    print("=" * 100)
    print("Generating audio segment:", segment["segment_number"])
    print("Speaker:", segment["speaker"])
    print("Voice:", segment["voice_name"])
    print("Text:", segment["script_text"])

    blob_name = (
        f"{GCS_AUDIO_PREFIX}/"
        f"{episode_id}/"
        f"{segment['segment_number']:02d}_{sanitize_slug(segment['speaker'])}.wav"
    )

    audio_gcs_uri = generate_tts_audio_segment(
        text=segment["script_text"],
        voice_name=segment["voice_name"],
        blob_name=blob_name,
    )

    completed_segment = {
        **segment,
        "audio_gcs_uri": audio_gcs_uri,
        "mime_type": "audio/wav",
        "tts_model": TTS_MODEL,
    }

    script_segments.append(completed_segment)

    print("Uploaded audio:", audio_gcs_uri)

print("Generated audio segments:", len(script_segments))

Generating audio segment: 1
Speaker: Maya
Voice: Kore
Text: Welcome, everyone. Today we're breaking down our internal 'Project Vox' initiative. The goal was to build a fully cloud-native pipeline to generate synthetic technical podcasts. Crucially, the entire system avoids any local media storage or manual processing, creating a repeatable, serverless, and scalable workflow.
Uploaded audio: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/audio/b533e3cd-4846-40c9-bceb-a401653df411/01_maya.wav
Generating audio segment: 2
Speaker: Leon
Voice: Puck
Text: That sounds powerful. So, where does it start? If I'm a developer, how do I get from my design documents to a finished audio package? Is the whole thing triggered by an API call?
Uploaded audio: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/audio/b533e3cd-4846-40c9-bceb-a401653df411/02_leon.wav
Generating audio segment: 3
Speaker: Maya
Voice: Kore
Text: It's event-driven. The process kicks off when a document is uploaded 

In [32]:
for segment in script_segments[:3]:
    print("=" * 100)
    print("Segment:", segment["segment_number"])
    print("Speaker:", segment["speaker"])
    print("GCS URI:", segment["audio_gcs_uri"])
    print("Text:", segment["script_text"])

    display_gcs_audio(segment["audio_gcs_uri"])

Segment: 1
Speaker: Maya
GCS URI: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/audio/b533e3cd-4846-40c9-bceb-a401653df411/01_maya.wav
Text: Welcome, everyone. Today we're breaking down our internal 'Project Vox' initiative. The goal was to build a fully cloud-native pipeline to generate synthetic technical podcasts. Crucially, the entire system avoids any local media storage or manual processing, creating a repeatable, serverless, and scalable workflow.


Segment: 2
Speaker: Leon
GCS URI: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/audio/b533e3cd-4846-40c9-bceb-a401653df411/02_leon.wav
Text: That sounds powerful. So, where does it start? If I'm a developer, how do I get from my design documents to a finished audio package? Is the whole thing triggered by an API call?


Segment: 3
Speaker: Maya
GCS URI: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/audio/b533e3cd-4846-40c9-bceb-a401653df411/03_maya.wav
Text: It's event-driven. The process kicks off when a document is uploaded to a designated Google Cloud Storage bucket. We use a Retrieval-Augmented Generation, or RAG, pattern. We don't feed whole documents to the model; we pre-process and index our knowledge base into searchable chunks first.


In [33]:
transcript_lines = []

for segment in script_segments:
    transcript_lines.append(
        f"{segment['segment_number']:02d}. {segment['speaker']}: {segment['script_text']}"
    )

episode_transcript = "\n\n".join(transcript_lines)

podcast_manifest = {
    "episode_id": episode_id,
    "episode_title": podcast_script_result["script_data"]["episode_title"],
    "episode_summary": podcast_script_result["script_data"]["episode_summary"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "tts_model": TTS_MODEL,
        "embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "audio_storage": {
        "bucket": BUCKET_NAME,
        "prefix": f"{GCS_AUDIO_PREFIX}/{episode_id}",
        "local_files_saved": False,
    },
    "segments": script_segments,
    "source_chunks_used": [
        {
            "chunk_id": result["chunk_id"],
            "title": result["title"],
            "topic": result["topic"],
            "source_type": result["source_type"],
            "distance": result["distance"],
        }
        for result in podcast_script_result["search_results"]
    ],
}

print("Transcript:")
print(episode_transcript[:1500])

print("\nManifest preview:")
print(json.dumps(podcast_manifest, indent=2, ensure_ascii=False, default=str)[:1800])

Transcript:
01. Maya: Welcome, everyone. Today we're breaking down our internal 'Project Vox' initiative. The goal was to build a fully cloud-native pipeline to generate synthetic technical podcasts. Crucially, the entire system avoids any local media storage or manual processing, creating a repeatable, serverless, and scalable workflow.

02. Leon: That sounds powerful. So, where does it start? If I'm a developer, how do I get from my design documents to a finished audio package? Is the whole thing triggered by an API call?

03. Maya: It's event-driven. The process kicks off when a document is uploaded to a designated Google Cloud Storage bucket. We use a Retrieval-Augmented Generation, or RAG, pattern. We don't feed whole documents to the model; we pre-process and index our knowledge base into searchable chunks first.

04. Leon: Okay, so you have small chunks of text. How does the system find the relevant ones for a specific podcast topic? How does it understand the *meaning* of the c

In [34]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

transcript_blob_name = (
    f"{GCS_TRANSCRIPT_PREFIX}/"
    f"{episode_id}/"
    f"transcript_{timestamp}.txt"
)

manifest_blob_name = (
    f"{GCS_MANIFEST_PREFIX}/"
    f"{episode_id}/"
    f"manifest_{timestamp}.json"
)

transcript_gcs_uri = upload_text_to_gcs(
    episode_transcript,
    blob_name=transcript_blob_name,
    content_type="text/plain",
)

manifest_gcs_uri = upload_text_to_gcs(
    json.dumps(
        podcast_manifest,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    blob_name=manifest_blob_name,
    content_type="application/json",
)

print("Transcript saved to:")
print(transcript_gcs_uri)

print("\nManifest saved to:")
print(manifest_gcs_uri)

Transcript saved to:
gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/transcripts/b533e3cd-4846-40c9-bceb-a401653df411/transcript_20260701_153418.txt

Manifest saved to:
gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/manifests/b533e3cd-4846-40c9-bceb-a401653df411/manifest_20260701_153418.json


In [35]:
asset_rows = [
    {
        "asset_id": str(uuid4()),
        "episode_id": episode_id,
        "asset_type": "transcript",
        "title": podcast_script_result["script_data"]["episode_title"],
        "gcs_uri": transcript_gcs_uri,
        "mime_type": "text/plain",
        "metadata_json": json.dumps(
            {
                "segment_count": len(script_segments),
                "local_files_saved": False,
            },
            ensure_ascii=False,
            default=str,
        ),
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "asset_id": str(uuid4()),
        "episode_id": episode_id,
        "asset_type": "manifest",
        "title": podcast_script_result["script_data"]["episode_title"],
        "gcs_uri": manifest_gcs_uri,
        "mime_type": "application/json",
        "metadata_json": json.dumps(
            {
                "segment_count": len(script_segments),
                "audio_segment_count": len(script_segments),
                "local_files_saved": False,
            },
            ensure_ascii=False,
            default=str,
        ),
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
]

for segment in script_segments:
    asset_rows.append(
        {
            "asset_id": str(uuid4()),
            "episode_id": episode_id,
            "asset_type": "audio_segment",
            "title": f"{podcast_script_result['script_data']['episode_title']} - segment {segment['segment_number']}",
            "gcs_uri": segment["audio_gcs_uri"],
            "mime_type": "audio/wav",
            "metadata_json": json.dumps(
                {
                    "segment_number": segment["segment_number"],
                    "speaker": segment["speaker"],
                    "voice_name": segment["voice_name"],
                    "source_chunk_ids": segment["source_chunk_ids"],
                    "local_files_saved": False,
                },
                ensure_ascii=False,
                default=str,
            ),
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )

print("Asset rows:", len(asset_rows))

Asset rows: 12


In [36]:
bigquery_client.query(
    f"TRUNCATE TABLE `{script_table_ref}`",
    location=dataset.location,
).result()

bigquery_client.query(
    f"TRUNCATE TABLE `{asset_table_ref}`",
    location=dataset.location,
).result()

script_errors = bigquery_client.insert_rows_json(
    script_table_ref,
    script_segments,
)

asset_errors = bigquery_client.insert_rows_json(
    asset_table_ref,
    asset_rows,
)

if script_errors:
    print("Script insert errors:")
    print(script_errors)
else:
    print("Inserted script rows:", len(script_segments))

if asset_errors:
    print("Asset insert errors:")
    print(asset_errors)
else:
    print("Inserted asset rows:", len(asset_rows))

Inserted script rows: 10
Inserted asset rows: 12


In [37]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{document_table_ref}`) AS document_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (SELECT COUNT(*) FROM `{script_table_ref}`) AS script_segment_count,
  (SELECT COUNT(*) FROM `{asset_table_ref}`) AS asset_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print("Document count:", result.document_count)
print("Chunk count:", result.chunk_count)
print("Script segment count:", result.script_segment_count)
print("Asset count:", result.asset_count)
print("Embedding length:", result.embedding_length)

Document count: 5
Chunk count: 66
Script segment count: 10
Asset count: 12
Embedding length: 768


In [38]:
sql = f"""
SELECT
  asset_type,
  title,
  gcs_uri,
  mime_type,
  created_at
FROM `{asset_table_ref}`
ORDER BY
  CASE asset_type
    WHEN 'manifest' THEN 1
    WHEN 'transcript' THEN 2
    WHEN 'audio_segment' THEN 3
    ELSE 4
  END,
  gcs_uri
"""

podcast_assets_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

podcast_assets_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,asset_type,title,gcs_uri,mime_type,created_at
0,manifest,Cloud-Native AI Podcast Pipeline,gs://leafy-guide-497515-m4-vector-assets/podca...,application/json,2026-07-01 15:34:27.579462+00:00
1,transcript,Cloud-Native AI Podcast Pipeline,gs://leafy-guide-497515-m4-vector-assets/podca...,text/plain,2026-07-01 15:34:27.579158+00:00
2,audio_segment,Cloud-Native AI Podcast Pipeline - segment 1,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.579821+00:00
3,audio_segment,Cloud-Native AI Podcast Pipeline - segment 2,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.579897+00:00
4,audio_segment,Cloud-Native AI Podcast Pipeline - segment 3,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.579986+00:00
5,audio_segment,Cloud-Native AI Podcast Pipeline - segment 4,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.580831+00:00
6,audio_segment,Cloud-Native AI Podcast Pipeline - segment 5,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.581034+00:00
7,audio_segment,Cloud-Native AI Podcast Pipeline - segment 6,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.581224+00:00
8,audio_segment,Cloud-Native AI Podcast Pipeline - segment 7,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.581306+00:00
9,audio_segment,Cloud-Native AI Podcast Pipeline - segment 8,gs://leafy-guide-497515-m4-vector-assets/podca...,audio/wav,2026-07-01 15:34:27.581494+00:00


In [39]:
def answer_podcast_source_question(
    question: str,
    *,
    top_k: int = 8,
) -> dict:
    results = search_podcast_chunks(
        question,
        top_k=top_k,
    )

    context = build_podcast_context(results)

    prompt = f"""
You are a technical podcast research assistant.

Answer the question using only the retrieved source chunks.

Question:
{question}

Retrieved chunks:
{context}

Return:
1. Short answer
2. Technical explanation
3. Suggested podcast talking points
4. Sources used, referencing SOURCE numbers

Do not invent information outside the chunks.
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "question": question,
        "answer": response.text,
        "search_results": results,
        "context": context,
    }


podcast_qa_result = answer_podcast_source_question(
    "How should we explain the difference between storing podcast audio in GCS and storing metadata plus embeddings in BigQuery?",
    top_k=8,
)

print(podcast_qa_result["answer"])

1.  **Short Answer**
    Google Cloud Storage (GCS) is used for storing large, unstructured binary assets like podcast audio files, raw source documents, and final manifest JSONs. BigQuery, on the other hand, is for structured data, specifically storing metadata about podcast episodes and text chunks, along with their high-dimensional vector embeddings to enable semantic search and Retrieval-Augmented Generation (RAG).

2.  **Technical Explanation**
    GCS functions as the primary data lake and durable object store for the podcast pipeline. It holds the actual audio segments (e.g., MP3 files) generated for the podcast, the raw source documents used as input, the generated script text, and the final JSON manifests that define the episode structure. Its role is to store these "files" or "assets" directly and durably [SOURCE 2, 3, 5, 6, 8].

    BigQuery serves a dual purpose. Firstly, it stores structured metadata about each podcast episode, including details like topic, status, script 

In [40]:
sql = f"""
SELECT
  topic,
  source_type,
  COUNT(*) AS chunk_count,
  ROUND(AVG(chunk_char_count), 2) AS avg_chunk_chars,
  MIN(chunk_char_count) AS min_chunk_chars,
  MAX(chunk_char_count) AS max_chunk_chars
FROM `{chunk_table_ref}`
GROUP BY
  topic,
  source_type
ORDER BY
  topic,
  chunk_count DESC
"""

chunk_analytics_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

chunk_analytics_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,topic,source_type,chunk_count,avg_chunk_chars,min_chunk_chars,max_chunk_chars
0,cloud_architecture,architecture_note,13,966.92,598,1088
1,cloud_execution,operations_guide,12,1033.75,729,1099
2,developer_education,lesson_notes,14,1043.50,947,1100
3,rag_and_chunking,engineering_design,15,1000.00,570,1099
4,structured_outputs,data_contract,12,1037.25,966,1099


In [41]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "63_w_google_cloud_podcast_ai_rag_pipeline.ipynb",
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "tts_model": TTS_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "document_table": document_table_ref,
        "chunk_table": chunk_table_ref,
        "script_table": script_table_ref,
        "asset_table": asset_table_ref,
    },
    "cloud_storage": {
        "audio_prefix": f"gs://{BUCKET_NAME}/{GCS_AUDIO_PREFIX}/{episode_id}",
        "transcript_gcs_uri": transcript_gcs_uri,
        "manifest_gcs_uri": manifest_gcs_uri,
        "local_files_saved": False,
    },
    "episode": {
        "episode_id": episode_id,
        "title": podcast_script_result["script_data"]["episode_title"],
        "summary": podcast_script_result["script_data"]["episode_summary"],
        "segment_count": len(script_segments),
    },
    "document_count": len(source_documents),
    "chunk_count": len(chunk_rows),
    "asset_count": len(asset_rows),
    "podcast_qa_result": podcast_qa_result,
    "chunk_analytics": chunk_analytics_df.to_dict(orient="records"),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{episode_id}/"
    f"summary_{timestamp}.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/summaries/b533e3cd-4846-40c9-bceb-a401653df411/summary_20260701_153418.json


In [42]:
print("Podcast AI RAG Pipeline notebook completed.")
print("=" * 100)

print("Episode ID:", episode_id)
print("Episode title:", podcast_script_result["script_data"]["episode_title"])
print("Documents:", len(source_documents))
print("Chunks:", len(chunk_rows))
print("Script segments:", len(script_segments))
print("Assets:", len(asset_rows))

print("\nImportant GCS URIs:")
print("Transcript:", transcript_gcs_uri)
print("Manifest:", manifest_gcs_uri)
print("Summary:", summary_gcs_uri)

print("\nBigQuery tables:")
print("-", document_table_ref)
print("-", chunk_table_ref)
print("-", script_table_ref)
print("-", asset_table_ref)

print("\nAudio segments:")

for segment in script_segments:
    print("=" * 100)
    print("Segment:", segment["segment_number"])
    print("Speaker:", segment["speaker"])
    print("Voice:", segment["voice_name"])
    print("Audio:", segment["audio_gcs_uri"])
    print("Text:", segment["script_text"])

print("\nNo local audio/video/image files were saved.")

Podcast AI RAG Pipeline notebook completed.
Episode ID: b533e3cd-4846-40c9-bceb-a401653df411
Episode title: Cloud-Native AI Podcast Pipeline
Documents: 5
Chunks: 66
Script segments: 10
Assets: 12

Important GCS URIs:
Transcript: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/transcripts/b533e3cd-4846-40c9-bceb-a401653df411/transcript_20260701_153418.txt
Manifest: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/manifests/b533e3cd-4846-40c9-bceb-a401653df411/manifest_20260701_153418.json
Summary: gs://leafy-guide-497515-m4-vector-assets/podcast-ai-rag/summaries/b533e3cd-4846-40c9-bceb-a401653df411/summary_20260701_153418.json

BigQuery tables:
- leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_documents
- leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_source_chunks
- leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_script_segments
- leafy-guide-497515-m4.podcast_ai_rag_pipeline.podcast_assets

Audio segments:
Segment: 1
Speaker: Maya
Voice: Kore
A